# M2 — Find and Fix the CET Bug
**Duration:** 45 min | **Participant notebook:** `M2_CET_BUG.ipynb`

---

## At a glance

| | |
|---|---|
| **Copilot skill taught** | Context sensitivity — no context vs. precise context; how the prompt determines the fix |
| **Key file** | `curve_pipeline/cet_converter.py` |
| **What participants produce** | A throwaway fix on branch `m2-exploration` (created and deleted by the notebook); **no permanent change to main** |

**Critical constraint:** `cet_converter.py` on `main` must stay buggy after M2. M4 TDD depends on it. The notebook creates and deletes the exploration branch — verify this happened before starting M3.

**Regression arc:**

| Checkpoint | Before fix | After summer fix | After order B fix |
|---|---|---|---|
| `checkpoint_winter.csv` | 0 | 0 | 0 |
| `checkpoint_summer.csv` | 75 | 0 | 0 |
| `checkpoint_october.csv` | 29 | 4 | 0 |

In production this was 29,211 → 6 → 0. Mechanism identical, scale different.

**Fresh chat requirement:** Tell participants this verbally before they start: open a new Copilot Chat panel, close all other notebooks, add no `#file:` references for Steps 1–2. The repo contains facilitator files that let Copilot find the bug immediately — context isolation is essential or the lesson disappears.

## Key teaching moment

**The difference between a bad fix and a good one is not Copilot's capability — it's the specificity of what you told it.**

Deliver this after Step 2 (no-context fix) and before Step 3 (precise context):

> "Look at what Copilot did with no context. Now compare it to what happens when you give it the domain rule: in summer, cetStarttime = startTime minus one hour, the helpers exist but aren't called, use them. Same model. Different output. The change is in the prompt, not in Copilot."

Secondary teaching moment — deliver before running the winter regression (Step 2):

> "Zero mismatches. Ship it, right? That's what happened for 10 months. What you're about to see is what February doesn't test."

Then run the summer checkpoint. Let the 75 mismatches land before saying anything.

## Questions and answers

**After Step 1 (explain with no context):**

Q: "Did Copilot mention that the CET helpers are never called?"

A: Almost never on first pass. It describes intent correctly — "this module converts startTime to CET" — without noticing that the conversion functions exist but aren't invoked. If a participant gets this on first pass, ask them to show the exact prompt they used. Worth examining.

---

**After Step 2 (fix with no context + regression):**

Q: "What did Copilot fix? Run the summer regression. What's the count now?"

A: Results vary. Common outcomes: still 75 (Copilot rewrote something cosmetic), reduced but non-zero (partial fix), zero summer but new fall-back failures (most instructive — sets up order B). Don't correct wrong fixes. Let the regression tell the story.

---

**After fall-back still has 4 mismatches (Step 4):**

Q: "Look at the failing rows. What do they have in common?"

A: All are Oct 25 or 26, specifically `order=B` rows at 02:00. The fall-back night: clock goes 02:00 → 02:00. Order A is still CEST; order B is already CET.

---

**Closing:**

Q: "If you only had the winter checkpoint, would this fix have shipped?"

A: Yes. That's the lesson. Winter regression would still be 0. The bug is invisible in February. That's exactly why M4 writes summer and fall-back tests.

## Watch for

**Copilot solves the bug immediately in Step 1 (fresh chat not opened).**

What it looks like: Step 1 ask (no context) returns a fully correct fix on the first try.

Response: Check whether they opened a new chat and closed other notebooks. The repo's facilitator files and other notebooks describe the bug. If context leaked, have them open a truly fresh chat (`Ctrl+Shift+P` → *New Chat*), close all other editor tabs, and retry. The comparison between Steps 2 and 3 is the module's lesson — without the Step 2 failure, it doesn't land.

---

**Copilot fixes summer but introduces a regression on fall-back order B.**

What it looks like: After Step 3, `checkpoint_summer.csv → 0`, but `checkpoint_october.csv` gets worse.

Response: Best possible outcome for the module. Don't fix it immediately. Ask: "Is this fix correct?" Make them articulate what went wrong. Say: "We're going to make sure this can't happen again. That's M4."

---

**Participants commit or push their fix to main.**

What it looks like: `git status` shows a commit on main, or they did `git push`.

Response: Run `git reset HEAD~1 --soft` and unstage. Do not let a fixed `cet_converter.py` survive to M4.

---

**At the break — verify all machines are clean:**

Run `git status` and `git log --oneline -3` on a visible screen. Confirm: branch is `main`, `cet_converter.py` is unchanged (still buggy), no pending commits.

## Timing notes

| Segment | Planned | Actual typical | Notes |
|---|---|---|---|
| Fresh chat setup + winter regression | 5 min | 6 min | |
| Summer regression reveal | 3 min | 3 min | Let the number land silently |
| Step 1 — explain, no context | 5 min | 5 min | Quick; debrief question is the value |
| Step 2 — fix, no context + regression | 8 min | 10 min | Participants iterate on partial fixes; let it run |
| Step 3 — fix, precise context | 8 min | 8 min | Prompt construction is the activity |
| Run both regressions after fix | 4 min | 4 min | |
| Step 4 — order B identification + fix | 8 min | 14 min | Oct 25–26 row inspection is slower than expected |
| Branch cleanup + debrief | 4 min | 4 min | |

**If you're running long:** Steps 4 (order B) is cuttable. Stop after summer is fixed, note the remaining 4 fall-back mismatches, say "The order B case is exactly what M4's TDD will protect." Clean handoff.

**If you're running short:** The bonus reflection — "What prompt would guarantee Copilot gets fall-back order B right on the first try?" — fills 5 minutes and previews M3.